In [0]:
import pandas as pd
import os
import sys
from datetime import date


sys.path.append("/Volumes/opsanalytics_adb_workspace01/default/oracle_connections")
from db_config import get_connection

In [0]:
conn = get_connection("production")
cursor = conn.cursor()

In [0]:
# Parameters
sched_start_date = "2021-01-01"
today = date.today()
sched_end_date = str(today)
status = "Completed"

# Table names
encounter_data_table_name = "MS_INSIGHT.OR_QUALITY_DASHBOARD_CASE_DETAILS"
room_master_data_table_name = "MS_INSIGHT.OR_QUALITY_DASHBOARD_ROOM_MASTER"
utlization_calculation_table = "MS_INSIGHT.OR_QUALITY_ROOM_UTIL_V"

In [0]:
or_quality_dashboard_case_details_query = f"""
    SELECT ENCOUNTERS.*
    FROM {encounter_data_table_name} ENCOUNTERS
    WHERE ENCOUNTERS.SURGERY_DATE >= TO_DATE('{sched_start_date}','YYYY-MM-DD')
      AND ENCOUNTERS.SURGERY_DATE <= TO_DATE('{sched_end_date}','YYYY-MM-DD')
"""

cursor = conn.cursor()
cursor.execute(or_quality_dashboard_case_details_query)
columns = [desc[0] for desc in cursor.description]
or_quality_dashboard_case_details = pd.DataFrame(cursor.fetchall(), columns=columns)

print(f"Rows returned: {len(or_quality_dashboard_case_details)}")

# Write a DataFrame to a specific catalog and schema
spark.createDataFrame(or_quality_dashboard_case_details).write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .format("delta") \
  .saveAsTable("opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details")

In [0]:
room_master_data_query = f"""
    SELECT *
    FROM {room_master_data_table_name} ROOM_MASTER
"""

cursor = conn.cursor()
cursor.execute(room_master_data_query)
columns = [desc[0] for desc in cursor.description]
room_master_data = pd.DataFrame(cursor.fetchall(), columns=columns)

print(f"Rows returned: {len(room_master_data)}")

# Write a DataFrame to a specific catalog and schema
spark.createDataFrame(room_master_data).write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .format("delta") \
  .saveAsTable("opsanalytics_adb_workspace01.or.or_quality_dashboard_room_master")

In [0]:
room_utilization_data_query = f"""
    SELECT *
    FROM {utlization_calculation_table}
"""

cursor = conn.cursor()
cursor.execute(room_utilization_data_query)
columns = [desc[0] for desc in cursor.description]
room_utilization_data = pd.DataFrame(cursor.fetchall(), columns=columns)

print(f"Rows returned: {len(room_utilization_data)}")

# Write a DataFrame to a specific catalog and schema
spark.createDataFrame(room_utilization_data).write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .format("delta") \
  .saveAsTable("opsanalytics_adb_workspace01.or.or_quality_room_utilization")